In [1]:
import polars as pl 
import polars_ds as pds 
import requests 
import json 
import duckdb
from pathlib import Path
from src.utils import scrape_ticket_sections, get_available_events, scrape_match_results, scrape_eliteserien_results,create_table_after_round

In [2]:
from importlib import reload
import src.utils as u
reload(u)

rows = u.scrape_eliteserien_all_xg_for_season(
season_id=2025,
year=2026,
delay_seconds=0
)

In [3]:
import polars as pl 
pl.DataFrame(rows).filter(pl.col('home_team') == 'SK Brann').tail(1)

season,date,matchday,home_team,away_team,result,sofascore_event_id,sofascore_url,home_xg,away_xg,home_xg_display,away_xg_display,home_xg_all,away_xg_all,home_xg_1st,away_xg_1st,home_xg_2nd,away_xg_2nd,home_xg_all_display,away_xg_all_display,home_xg_1st_display,away_xg_1st_display,home_xg_2nd_display,away_xg_2nd_display,snapshot_at,home_ballpossession,away_ballpossession,home_kilometerscovered,away_kilometerscovered,home_bigchancecreated,away_bigchancecreated,home_totalshotsongoal,away_totalshotsongoal,home_goalkeepersaves,away_goalkeepersaves,home_numberofsprints,away_numberofsprints,…,away_bigchancecreated_1st,home_totalshotsongoal_1st,away_totalshotsongoal_1st,home_goalkeepersaves_1st,away_goalkeepersaves_1st,home_cornerkicks_1st,away_cornerkicks_1st,home_fouls_1st,away_fouls_1st,home_passes_1st,away_passes_1st,home_totaltackle_1st,away_totaltackle_1st,home_freekicks_1st,away_freekicks_1st,home_yellowcards_1st,away_yellowcards_1st,home_ballpossession_2nd,away_ballpossession_2nd,home_bigchancecreated_2nd,away_bigchancecreated_2nd,home_totalshotsongoal_2nd,away_totalshotsongoal_2nd,home_goalkeepersaves_2nd,away_goalkeepersaves_2nd,home_cornerkicks_2nd,away_cornerkicks_2nd,home_fouls_2nd,away_fouls_2nd,home_passes_2nd,away_passes_2nd,home_totaltackle_2nd,away_totaltackle_2nd,home_freekicks_2nd,away_freekicks_2nd,home_yellowcards_2nd,away_yellowcards_2nd
i64,date,i64,str,str,str,i64,str,f64,f64,str,str,f64,f64,f64,f64,f64,f64,str,str,str,str,str,str,"datetime[μs, UTC]",i64,i64,f64,f64,i64,i64,i64,i64,i64,i64,i64,i64,…,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
2026,2026-09-05,20,"""SK Brann""","""Lillestrøm SK""","""1:2""",15260891,"""https://www.sofascore.com/no/f…",1.97,1.33,"""1.97""","""1.33""",1.97,1.33,0.83,0.85,1.14,0.48,"""1.97""","""1.33""","""0.83""","""0.85""","""1.14""","""0.48""",2026-09-09 18:01:51.636607 UTC,73,27,111.94,111.27,1,4,24,8,2,2,85,85,…,3,8,4,0,0,2,0,6,7,303,161,6,8,7,6,1,0,80,20,1,1,16,4,2,2,10,0,5,1,376,106,5,16,1,5,0,1


In [2]:
# List all table names
db_path = Path("data/brann.duckdb")
con = duckdb.connect(str(db_path))
table_names = con.execute("SELECT table_name FROM information_schema.tables WHERE table_schema = 'main'").fetchall()
print("Available tables:")
for table in table_names:
    print(f"  - {table[0]}")
con.close()

Available tables:
  - dim_teams
  - fct_goal_scorers
  - fct_league_standings
  - fct_matches
  - raw_eliteserien_goal_scorers
  - raw_eliteserien_results


In [25]:
con = duckdb.connect('data/brann.duckdb')
goal_scorers = pl.from_arrow(con.execute(
    """
    SELECT *
FROM fct_goal_scorers
""").arrow())
con.close()
goal_scorers


season,date,matchday,home_team,away_team,result,scorer_team,scorer_name,ingested_at
i64,date,i64,str,str,str,str,str,datetime[μs]
2015,2015-04-06,1,"""Sandefjord""","""Bodø/Glimt""","""3:1""","""FK Bodø/Glimt""","""H. Furebotn""",2026-09-04 14:29:08.523745
2015,2015-04-06,1,"""Mjøndalen""","""Viking FK""","""1:0""","""Mjøndalen IF""","""S. Kapidzic""",2026-09-04 14:29:08.523745
2015,2015-04-06,1,"""Sandefjord""","""Bodø/Glimt""","""3:1""","""Sandefjord Fotball""","""A. Gabrielsen""",2026-09-04 14:29:08.523745
2015,2015-04-06,1,"""Sandefjord""","""Bodø/Glimt""","""3:1""","""Sandefjord Fotball""","""J. Mendy""",2026-09-04 14:29:08.523745
2015,2015-04-06,1,"""Sandefjord""","""Bodø/Glimt""","""3:1""","""Sandefjord Fotball""","""J. Mendy""",2026-09-04 14:29:08.523745
…,…,…,…,…,…,…,…,…
2026,2026-09-06,20,"""Molde FK""","""KFUM Oslo""","""2:0""","""Molde FK""","""T. Koné-Doherty""",2026-09-08 09:13:42.536941
2026,2026-09-06,20,"""Sarpsborg 08""","""Vålerenga""","""2:2""","""Sarpsborg 08 FF""","""C. Niyukuri""",2026-09-08 09:13:42.536941
2026,2026-09-06,20,"""Sarpsborg 08""","""Vålerenga""","""2:2""","""Sarpsborg 08 FF""","""S. Sørli""",2026-09-08 09:13:42.536941


In [2]:
con = duckdb.connect('data/brann.duckdb')
latest = pl.from_arrow(con.execute("""
    SELECT * FROM fct_league_standings 
""").arrow())
con.close()

latest

season,matchday,team,total_points,total_goals_for,total_goals_against,goal_difference,position
i64,i64,str,"decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]",i64
2026,19,"""Bodø/Glimt""",44,47,14,33,1
2026,19,"""Viking FK""",43,42,18,24,2
2026,19,"""Tromsø IL""",35,34,20,14,3
2026,19,"""Molde FK""",30,36,29,7,4
2026,19,"""SK Brann""",26,36,27,9,5
…,…,…,…,…,…,…,…
2015,1,"""Viking FK""",0,0,1,-1,12
2015,1,"""Tromsø IL""",0,0,1,-1,13
2015,1,"""Bodø/Glimt""",0,1,3,-2,14


In [3]:


# Connect to DuckDB and read raw_eliteserien_results
db_path = Path("data/brann.duckdb")
con = duckdb.connect(str(db_path))
eliteserien_db = pl.from_arrow(con.execute("SELECT * FROM dim_teams").arrow())
con.close()

print(f"Loaded {len(eliteserien_db)} records from DuckDB")
eliteserien_db

Loaded 192 records from DuckDB


season,team_name
i64,str
2015,"""Aalesunds FK"""
2015,"""Bodø/Glimt"""
2015,"""Haugesund"""
2015,"""IK Start"""
2015,"""Lillestrøm SK"""
…,…
2026,"""Sandefjord"""
2026,"""Sarpsborg 08"""
2026,"""Tromsø IL"""


In [ ]:
from src.config import ELITESERIEN_SEASONS, SCRAPE_DELAY_SECONDS
from src.utils import scrape_eliteserien_goal_scorers_for_seasons

goal_scorers = scrape_eliteserien_goal_scorers_for_seasons(
    seasons=[(2014,2015)],
    delay_seconds=1,
)

processing match 1 of 240
processing match 2 of 240
processing match 3 of 240
processing match 4 of 240
processing match 5 of 240
processing match 6 of 240
processing match 7 of 240
processing match 8 of 240
processing match 9 of 240


In [3]:
import polars as pl
pl.DataFrame(goal_scorers)

season,date,matchday,home_team,away_team,result,scorer_team,scorer_name
i64,date,i64,str,str,str,str,str
2015,2015-04-06,1,"""Mjøndalen""","""Viking FK""","""1:0""","""Mjøndalen IF""","""S. Kapidzic"""
2015,2015-04-06,1,"""Rosenborg BK""","""Aalesunds FK""","""5:0""",""" ""","""P. Helland"""
2015,2015-04-06,1,"""Rosenborg BK""","""Aalesunds FK""","""5:0""",""" ""","""P. Helland"""
2015,2015-04-06,1,"""Rosenborg BK""","""Aalesunds FK""","""5:0""",""" ""","""A. Søderlund"""
2015,2015-04-06,1,"""Rosenborg BK""","""Aalesunds FK""","""5:0""",""" ""","""A. Søderlund"""
…,…,…,…,…,…,…,…
2026,2026-08-30,19,"""Lillestrøm SK""","""Fredrikstad FK""","""1:4""","""Lillestrøm SK""","""F. Gulbrandsen"""
2026,2026-08-30,19,"""Lillestrøm SK""","""Fredrikstad FK""","""1:4""","""Fredrikstad FK""","""S. Owusu"""
2026,2026-08-30,19,"""Lillestrøm SK""","""Fredrikstad FK""","""1:4""","""Fredrikstad FK""","""M. Nilsson"""


In [2]:
available_events = get_available_events()

In [3]:
for event in available_events:
    print(event['event_id'])


1085523
1187151
1188514


In [18]:
eliteserien_results = scrape_eliteserien_results(season_id = 2025,year = 2026)

In [19]:
pl.DataFrame(eliteserien_results)

date,matchday,home_team,away_team,result,snapshot_at
date,i64,str,str,str,"datetime[μs, UTC]"
2026-03-14,1,"""HamKam""","""Viking FK""","""2:1""",2026-09-01 09:53:45.721672 UTC
2026-03-14,1,"""Molde FK""","""Rosenborg BK""","""2:0""",2026-09-01 09:53:45.721672 UTC
2026-03-15,1,"""Kristiansund BK""","""SK Brann""","""3:2""",2026-09-01 09:53:45.721672 UTC
2026-03-15,1,"""KFUM Oslo""","""IK Start""","""2:0""",2026-09-01 09:53:45.721672 UTC
2026-03-15,1,"""Vålerenga""","""Sandefjord""","""1:0""",2026-09-01 09:53:45.721672 UTC
…,…,…,…,…,…
2026-08-30,19,"""IK Start""","""KFUM Oslo""","""4:1""",2026-09-01 09:53:45.721672 UTC
2026-08-30,19,"""Tromsø IL""","""Sarpsborg 08""","""0:0""",2026-09-01 09:53:45.721672 UTC
2026-08-30,19,"""Viking FK""","""Aalesunds FK""","""2:1""",2026-09-01 09:53:45.721672 UTC


In [3]:
create_table_after_round(eliteserien_results)

matchday,team,points,goals_for,goals_against,goal_difference,total_points,total_goals_for,total_goals_against,total_goal_difference,table_position
i64,str,i32,i64,i64,i64,i32,i64,i64,i64,u32
1,"""HamKam""",3,2,1,1,3,2,1,1,1
1,"""KFUM Oslo""",3,2,0,2,3,2,0,2,1
1,"""Kristiansund BK""",3,3,2,1,3,3,2,1,1
1,"""Lillestrøm SK""",3,3,1,2,3,3,1,2,1
1,"""Molde FK""",3,2,0,2,3,2,0,2,1
…,…,…,…,…,…,…,…,…,…,…
18,"""KFUM Oslo""",1,1,1,0,19,19,27,-8,12
18,"""Sandefjord""",3,2,1,1,18,15,23,-8,13
18,"""Aalesunds FK""",1,5,5,0,15,27,41,-14,14


In [4]:
paok = scrape_ticket_sections(1187151)

In [8]:
from src.utils import scrape_eliteserien_results_for_seasons

results = scrape_eliteserien_results_for_seasons(
    seasons=[
        (2025, 2026),
        (2024, 2025),
        (2023, 2024),
    ],
    delay_seconds=5.0,
)

In [10]:
pl.DataFrame(results).filter(pl.col('date').dt.year()==2025).filter(pl.col('home_team').str.contains('Brann'))

date,matchday,home_team,away_team,result,snapshot_at
date,i64,str,str,str,"datetime[μs, UTC]"
2025-04-06,2,"""SK Brann""","""Tromsø IL""","""3:1""",2026-09-02 11:00:27.335750 UTC
2025-04-10,17,"""SK Brann""","""Strømsgodset""","""2:1""",2026-09-02 11:00:27.335750 UTC
2025-04-27,4,"""SK Brann""","""Bryne""","""3:2""",2026-09-02 11:00:27.335750 UTC
2025-05-11,6,"""SK Brann""","""Rosenborg BK""","""0:0""",2026-09-02 11:00:27.335750 UTC
2025-05-16,7,"""SK Brann""","""Sarpsborg 08""","""2:2""",2026-09-02 11:00:27.335750 UTC
…,…,…,…,…,…
2025-09-28,19,"""SK Brann""","""Fredrikstad FK""","""1:0""",2026-09-02 11:00:27.335750 UTC
2025-10-18,25,"""SK Brann""","""Haugesund""","""4:1""",2026-09-02 11:00:27.335750 UTC
2025-10-29,23,"""SK Brann""","""Bodø/Glimt""","""1:2""",2026-09-02 11:00:27.335750 UTC


In [2]:
from src.agent import run_question

run_question("Hva er den lengste seiersstreaken som Brann har hatt i 2026?")

SQL:
WITH brann_matches AS (
    SELECT
        date,
        matchday,
        season,
        home_team,
        away_team,
        winner
    FROM fct_matches
    WHERE season = 2026
      AND (home_team = 'SK Brann' OR away_team = 'SK Brann')
    ORDER BY date
),
streaks AS (
    SELECT
        date,
        matchday,
        CASE
            WHEN (home_team = 'SK Brann' AND winner = 'home_team')
              OR (away_team = 'SK Brann' AND winner = 'away_team')
            THEN 1 ELSE 0 END AS is_win
    FROM brann_matches
    ORDER BY date
),
grouped AS (
    SELECT
        *,
        SUM(CASE WHEN is_win = 0 THEN 1 ELSE 0 END) OVER (ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS grp
    FROM streaks
),
win_streaks AS (
    SELECT
        grp,
        COUNT(*) AS streak_length
    FROM grouped
    WHERE is_win = 1
    GROUP BY grp
)
SELECT
    COALESCE(MAX(streak_length), 0) AS longest_win_streak
FROM win_streaks

Sammendrag:
Den lengste seiersrekken Brann har

In [ ]:

import src.agent
for event in src.agent.agent.stream(
    {"question": ""},
    stream_mode="updates",
):
    print(event)

{'plan_tasks': {'tasks': ['Finn Branns form de siste kampene før møtet med Start i 2026.', 'Finn Starts form de siste kampene før møtet med Brann i 2026.', 'Beregn og presenter målforskjellen for Brann i de siste kampene før møtet.', 'Beregn og presenter målforskjellen for Start i de siste kampene før møtet.'], 'task_index': 0, 'generated_sqls': [], 'collected_results': [], 'current_task': 'Finn Branns form de siste kampene før møtet med Start i 2026.', 'sql_error': '', 'attempts': 0}}
{'generate_sql': {'current_task': 'Finn Branns form de siste kampene før møtet med Start i 2026.', 'sql': "WITH start_team AS (\n    SELECT team_name\n    FROM dim_teams\n    WHERE season = 2026\n      AND team_name ILIKE '%Start%'\n    LIMIT 1\n), start_match AS (\n    SELECT date\n    FROM fct_matches\n    WHERE season = 2026\n      AND ((home_team = 'SK Brann' AND away_team = (SELECT team_name FROM start_team))\n        OR (away_team = 'SK Brann' AND home_team = (SELECT team_name FROM start_team)))\n 

In [6]:
query = """

WITH brann_matches AS (
    SELECT
        date,
        matchday,
        season,
        CASE
            WHEN home_team = 'SK Brann' THEN 'home'
            ELSE 'away'
        END AS venue,
        winner
    FROM fct_matches
    WHERE season = 2026
      AND (home_team = 'SK Brann' OR away_team = 'SK Brann')
    ORDER BY date
),
streaks AS (
    SELECT
        date,
        matchday,
        season,
        winner,
        SUM(
            CASE
                WHEN winner =
                    CASE
                        WHEN venue = 'home' THEN 'home_team'
                        ELSE 'away_team'
                    END
                THEN 0
                ELSE 1
            END
        ) OVER (
            ORDER BY date
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS grp
    FROM brann_matches
),
win_streaks AS (
    SELECT
        grp,
        COUNT(*) AS streak_length
    FROM (
        SELECT
            *,
            CASE
                WHEN winner =
                    CASE
                        WHEN venue = 'home' THEN 'home_team'
                        ELSE 'away_team'
                    END
                THEN 1
                ELSE 0
            END AS is_win
        FROM brann_matches
    ) AS t
    CROSS JOIN LATERAL (
        SELECT
            SUM(
                CASE WHEN is_win = 0 THEN 1 ELSE 0 END
            ) OVER (
                ORDER BY date
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) AS grp
    )
    WHERE is_win = 1
    GROUP BY grp
)
SELECT MAX(streak_length) AS longest_win_streak
FROM win_streaks;
"""

con = duckdb.connect('data/brann.duckdb')
goal_scorers = pl.from_arrow(con.execute(query).arrow())
con.close()
goal_scorers

BinderException: Binder Error: LATERAL join cannot contain window functions!

In [2]:
from openai import OpenAI

client = OpenAI()

for model in client.models.list().data:
    print(model.id)

gpt-4-0613
gpt-4
gpt-3.5-turbo
gpt-6-astra
gpt-realtime-2.1
gpt-realtime-2.1-mini
gpt-transcribe
gpt-live-transcribe
davinci-002
babbage-002
gpt-3.5-turbo-instruct
gpt-3.5-turbo-instruct-0914
gpt-3.5-turbo-1106
tts-1-hd
tts-1-1106
tts-1-hd-1106
text-embedding-3-small
text-embedding-3-large
gpt-3.5-turbo-0125
gpt-4-turbo
gpt-4-turbo-2024-04-09
gpt-4o
gpt-4o-2024-05-13
gpt-4o-mini-2024-07-18
gpt-4o-mini
gpt-4o-2024-08-06
omni-moderation-latest
omni-moderation-2024-09-26
o1-2024-12-17
o1
o3-mini
o3-mini-2025-01-31
gpt-4o-2024-11-20
gpt-4o-mini-search-preview-2025-03-11
gpt-4o-mini-search-preview
gpt-4o-transcribe
gpt-4o-mini-transcribe
o1-pro-2025-03-19
o1-pro
gpt-4o-mini-tts
o3-2025-04-16
o4-mini-2025-04-16
o3
o4-mini
gpt-4.1-2025-04-14
gpt-4.1
gpt-4.1-mini-2025-04-14
gpt-4.1-mini
gpt-4.1-nano-2025-04-14
gpt-4.1-nano
gpt-image-1
gpt-4o-transcribe-diarize
gpt-5-chat-latest
gpt-5-2025-08-07
gpt-5
gpt-5-mini-2025-08-07
gpt-5-mini
gpt-5-nano-2025-08-07
gpt-5-nano
gpt-audio-2025-08-28
gpt-rea

In [4]:
client.models.list().data

[Model(id='gpt-4-0613', created=1686588896, object='model', owned_by='openai', shutdown_date='2026-10-23'),
 Model(id='gpt-4', created=1687882411, object='model', owned_by='openai', shutdown_date='2026-10-23'),
 Model(id='gpt-3.5-turbo', created=1677610602, object='model', owned_by='openai', shutdown_date='2026-10-23'),
 Model(id='gpt-6-astra', created=1787853604, object='model', owned_by='system', shutdown_date=None),
 Model(id='gpt-realtime-2.1', created=1782254687, object='model', owned_by='system', shutdown_date=None),
 Model(id='gpt-realtime-2.1-mini', created=1782254706, object='model', owned_by='system', shutdown_date=None),
 Model(id='gpt-transcribe', created=1785168027, object='model', owned_by='system', shutdown_date=None),
 Model(id='gpt-live-transcribe', created=1785168034, object='model', owned_by='system', shutdown_date=None),
 Model(id='davinci-002', created=1692634301, object='model', owned_by='system', shutdown_date='2026-09-28'),
 Model(id='babbage-002', created=16926